# Deprem Yol Hasar Tespiti Modeli Eğitimi

Bu notebook, bilgisayarınızda (lokalde) yapay zeka modelini eğitmek uzun süreceği için Google Colab üzerinde ücretsiz GPU kullanarak model eğitimini gerçekleştirmek için hazırlanmıştır.

**ÖNEMLİ:** Başlamadan önce üst menüden `Çalışma Zamanı (Runtime)` -> `Çalışma zamanı türünü değiştir (Change runtime type)` seçeneğine tıklayarak **Donanım hızlandırıcı (Hardware accelerator)** olarak **T4 GPU**'yu seçtiğinizden emin olun.

## 1. Gerekli Kütüphanelerin Kurulumu

In [ ]:
!pip install datasets opencv-python numpy tqdm scikit-learn torch torchvision segmentation-models-pytorch albumentations

## 2. Gerekli Dizinlerin Oluşturulması

In [ ]:
import os
os.makedirs('data/images/', exist_ok=True)
os.makedirs('data/masks/', exist_ok=True)
os.makedirs('data/tiles/images/', exist_ok=True)
os.makedirs('data/tiles/masks/', exist_ok=True)
os.makedirs('models/', exist_ok=True)
print("Klasörler başarıyla oluşturuldu!")

## 3. Veri İndirme İşlemi (`01_data_download.py`)
Lokaldeki kodun test amaçlı `num_samples=20` sınırı kaldırılmıştır, tüm veri seti indirilecek (1-2 dakika sürebilir).

In [ ]:
from datasets import load_dataset
import os

print("Veri seti çekiliyor (CSCRS/kate-cd)...")
try:
    dataset = load_dataset("CSCRS/kate-cd")
except Exception as e:
    print(f"Hata: {e}. İsim 'hkayabilisim/kate-cd' deneniyor.")
    dataset = load_dataset("hkayabilisim/kate-cd")

img_dir = 'data/images/'
mask_dir = 'data/masks/'

print("Resimler kaydediliyor...")
for i, veri in enumerate(dataset['train']):
    if i % 100 == 0:
        print(f"{i} adet resim işlendi...")
    
    post_img = veri.get('post_image') or veri.get('image')
    label_img = veri.get('label') or veri.get('mask')
    
    if post_img and label_img:
        post_img.save(os.path.join(img_dir, f"deprem_{i}.png"))
        label_img.save(os.path.join(mask_dir, f"maske_{i}.png"))

print(f"İşlem TAMAMLANDI! Resimler '{img_dir}' dizininde.")

## 4. Veri Parçalama / Hazırlama (`02_data_preparation.py`)

In [ ]:
import cv2
import os
from tqdm import tqdm
import numpy as np

img_dir = 'data/images/'
mask_dir = 'data/masks/'

out_img_dir = 'data/tiles/images/'
out_mask_dir = 'data/tiles/masks/'

tile_size = 512
print("🔪 Dilimleme işlemi başladı, sabırla bekle...")

all_images = os.listdir(img_dir)
for img_name in tqdm(all_images):
    img_path = os.path.join(img_dir, img_name)
    mask_path = os.path.join(mask_dir, img_name.replace('deprem_', 'maske_'))
    
    if not os.path.exists(mask_path):
        continue
        
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask is None:
        print(f"Hata: {img_name} okunamadı.")
        continue
        
    h, w, _ = img.shape
    
    for y in range(0, h, tile_size):
        for x in range(0, w, tile_size):
            img_tile = img[y:y+tile_size, x:x+tile_size]
            mask_tile = mask[y:y+tile_size, x:x+tile_size]
            
            if img_tile.shape[0] != tile_size or img_tile.shape[1] != tile_size:
                continue
                
            # Eğer maske tamamen siyahsa (hasar yoksa) bazılarını atla (dengesizliği önlemek için)
            if np.sum(mask_tile) == 0 and np.random.rand() > 0.1:
                continue
                
            base_name = os.path.splitext(img_name)[0]
            tile_name = f"{base_name}_y{y}_x{x}.png"
            
            cv2.imwrite(os.path.join(out_img_dir, tile_name), img_tile)
            cv2.imwrite(os.path.join(out_mask_dir, tile_name), mask_tile)

print("✅ İşlem bitti! Parçalar oluşturuldu.")

## 5. Model Eğitimi (`03_train.py`)
Model eğitilecek ve `/models/en_iyi_model.pth` olarak kaydedilecektir.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
from sklearn.model_selection import train_test_split
import albumentations as A
import segmentation_models_pytorch as smp
import torch.optim as optim
from tqdm import tqdm

class DepremDataset(Dataset):
    def __init__(self, file_list, img_dir, mask_dir, transform=None):
        self.file_list = file_list
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_name = self.file_list[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, 0)
        mask = np.where(mask > 0, 1, 0).astype(np.float32)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()

        return image, mask

tile_img_dir = 'data/tiles/images/'
tile_mask_dir = 'data/tiles/masks/'
save_path = 'models/en_iyi_model.pth'

all_tiles = sorted(os.listdir(tile_img_dir))
if len(all_tiles) == 0:
    print("Hata: Eğitim verisi bulunamadı!")
else:
    train_files, val_files = train_test_split(all_tiles, test_size=0.2, random_state=42)
    
    train_transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                 scale=(0.95, 1.05),
                 rotate=(-15, 15),
                 p=0.5),
        A.RandomBrightnessContrast(p=0.2),
    ])

    train_dataset = DepremDataset(train_files, tile_img_dir, tile_mask_dir, transform=train_transform)
    val_dataset = DepremDataset(val_files, tile_img_dir, tile_mask_dir)

    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Kullanılan cihaz: {device}")
    
    model = smp.Segformer(encoder_name="mit_b3", encoder_weights="imagenet", in_channels=3, classes=1)
    model.to(device)

    criterion = smp.losses.DiceLoss(mode='binary')
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

    epochs = 15
    best_loss = float('inf')

    print(f"🔥 Eğitim başlıyor! {epochs} tur...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

        for images, masks in pbar:
            images, masks = images.to(device), masks.to(device).unsqueeze(1).float()
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            pbar.set_postfix({'loss': f"{loss.item():.4f}"})

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device).unsqueeze(1).float()
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        print(f"👉 Epoch {epoch+1} Özet | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✨ Yeni en iyi model kaydedildi: {save_path}")
    
    print("Eğitim tamamlandı!")

## 6. Eğitilmiş Modeli İndirme
Eğitim bittikten sonra aşağıdaki hücreyi çalıştırarak çıkan **en_iyi_model.pth** dosyasını bilgisayarınıza indirebilirsiniz.

In [ ]:
from google.colab import files
import os

model_yolu = 'models/en_iyi_model.pth'
if os.path.exists(model_yolu):
    print(f"{model_yolu} dosyası indiriliyor...")
    files.download(model_yolu)
else:
    print("Model dosyası bulunamadı, eğitim tamamlanmamış olabilir.")